## Install the model

In [ ]:
# Clone the specific branch
!git clone --branch colab_version https://github.com/cordutie/SCAPES.git

# Move into repo root
%cd SCAPES

# Install dependencies (if you have requirements.txt)
!pip install -r requirements.txt

# Ensure Python can find the SCAPES package
import sys, os
repo_path = os.getcwd()
if repo_path not in sys.path:
    sys.path.append(repo_path)

print("✅ SCAPES setup complete. You can now do: from SCAPES import ...")

## Mount your Google Drive (optional)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# **Inference**

SCAPES model has many modes of inference. This notebook only has examples for "semantic" reconstruction from the dataset.

## **1. Importing stuff**

In [ ]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2
 
import torch
import json
from pathlib import Path
from IPython.display import Audio, display

from SCAPES.auxiliar.encodec_wrapper import EncodecProcessor
from SCAPES.models.factorization import load_local_encoder, load_global_encoder
from SCAPES.models.flow import load_flow_model
from SCAPES.auxiliar.clap_wrapper import CLAPWrapper
from SCAPES.inference.FlowInference import FlowInference, load_and_encode, run_interpolation_pipeline, run_resynthesis_pipeline

# Setup Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Default configurations
segment_length, context_length, hop_size = 5, 5, 1

# Initialize processor
processor_48k = EncodecProcessor(sr=48000, streamable=True, device=device)

## **2. Load your model and initialize the inference engine**

Point to the model checkpoint you want to use.

In [ ]:
# Flow model path
model_path = Path("path/to/your/model")  # Make sure this is the same as your training model path! <---------------------------

# Create paths for best models, models at epoch given and configs
best_flow_ckpt_path  = model_path / "checkpoints" / "best_flow_model.pt"
flow_config_path     = model_path / "checkpoints" / "flow_model_config.json"
best_local_ckpt_path = model_path / "checkpoints" / "best_local_encoder.pt"
local_config_path    = model_path / "checkpoints" / "local_encoder_config.json"

# Load checkpoints and configs
best_local_encoder = load_local_encoder(
    checkpoint_path=best_local_ckpt_path, 
    json_path=local_config_path, 
    device=device
)
best_flow_model = load_flow_model(
    checkpoint_path=best_flow_ckpt_path, 
    json_path=flow_config_path, 
    device=device
)
clap_model = CLAPWrapper(version="2023", use_cuda=True)

# Open the JSON to pull the exact geometry the model was trained on
with open(flow_config_path, 'r') as f:
    f_config = json.load(f)

# Load the inference engine that puts everything together. This is the main entry point for inference, and the one you will use for interpolation and resynthesis pipelines.
best_engine = FlowInference(
    model            = best_flow_model,
    local_encoder    = best_local_encoder,
    processor        = processor_48k,
    context_model    = clap_model,  # Passing clap or proxy
    segment_length   = segment_length,
    context_length   = context_length,
    atoms_frames     = f_config.get("frames_per_atom", 48),
    atoms_hop_frames = f_config.get("atoms_hop_frames", 15),
    crossfade_frames = f_config.get("crossfade_frames", 3),
    device           = device,
    verbose          = True
)

## **3. Inference examples**

## **3.1. Audio reconstruction**

Point to a particular audio file in your dataset and reconstruct it from its full latent representation.

In [ ]:
duration = 10 # You can reduce the duration of the audio generated if it takes too long to generate the audio

audio_path   = "path/to/your/audiofile.wav"  # Change this to the path of your audio file <---------------------------
audio_tensor = best_engine.load_audio_to_tensor(audio_path)[:, :, :duration * 48000] # Shape [1, 2, T]
filename     = Path(audio_path).stem
print("Original:    ", filename)
display(Audio(audio_tensor.detach().cpu().numpy()[0], rate=48000))

# Run resynthesis pipeline
output = run_resynthesis_pipeline(
    engine     = best_engine,
    audio_path = audio_path,
    duration   = duration, 
    play       = True,
    TF         = True,
    NFE        = 32, # Number of function evaluations can be reduced to speed up inference at the cost of some audio quality.
)

## **3.2. Semantic reconstruction**

Point to a particular audio file in your dataset and reconstruct it from its semantics only.

In [ ]:
duration = 10 # You can reduce the duration of the audio generated if it takes too long to generate the audio

audio_path   = "path/to/your/audiofile.wav"  # Change this to the path of your audio file <---------------------------
audio_tensor = best_engine.load_audio_to_tensor(audio_path)[:, :, :duration * 48000] # Shape [1, 2, T]
filename     = Path(audio_path).stem
print("Original:    ", filename)
display(Audio(audio_tensor.detach().cpu().numpy()[0], rate=48000))

output = run_resynthesis_pipeline(
    engine     = best_engine,
    audio_path = audio_path,
    duration   = duration,
    play       = True,
    TF         = "partial", # Start with a little help from the audio data. Drops the help after 0.5 seconds.
    NFE        = 32, # Number of function evaluations can be reduced to speed up inference at the cost of some audio quality.
)

## **3.3. Semantic interpolation**

Point to two audio files and explore what is in the middle of them. You can choose between linear and spherical interpolation for the semantic embeddings interpolation.

In [ ]:
audio_start_path   = "path/to/your/audiofile.wav"  # Change this to the path of your audio file <---------------------------
audio_start_tensor = best_engine.load_audio_to_tensor(audio_start_path)[:,:,:5*48000] # Shape [1, 2, T]
filename     = Path(audio_start_path).stem
print("Original_start:    ", filename)
display(Audio(audio_start_tensor.detach().cpu().numpy()[0], rate=48000))

audio_end_path   = "path/to/your/audiofile.wav"  # Change this to the path of your audio file <---------------------------
audio_end_tensor = best_engine.load_audio_to_tensor(audio_end_path)[:,:,:5*48000] # Shape [1, 2, T]
filename     = Path(audio_end_path).stem
print("Original_end:     ", filename)
display(Audio(audio_end_tensor.detach().cpu().numpy()[0], rate=48000))

output = run_interpolation_pipeline(
    engine         = best_engine,
    audio_path_1   = audio_start_path,
    audio_path_2   = audio_end_path,
    timeline_size  = 200, # Let's go for around 10 seconds
    stay_time      = 20,  # We will start with around 2 seconds of the first and end with around 2 seconds of the second, leaving around 8 seconds for the interpolation in the middle
    stickyness     = 3.0, # Adjust this to decide how sticky is the middle point between the two audios. 1.0 is linear, higher is stickier and lower is less sticky.
    plot_stickyness_curve = True,
    play           = True,
    NFE            = 32, # Number of function evaluations can be reduced to speed up inference at the cost of some audio quality.
    context_static = False,  # If True, uses the first context embedding for each audio only
)